# Milestone 2 lab, notebook 3 of 8: A believable value

## Where we are

| Notebook | What you do |
|---|---|
| 1. Empty in which rows | Write a check for empty cells that does not fire on good rows |
| 2. Unique on which columns | Find the receipt and the file that were counted twice |
| **3. A believable value** (you are here) | Catch 1,180 kg of bananas without catching 400 real bottles |
| 4. Rules of the business | Three rules a person must tell you. Each needs a join first |
| 5. Across two tables | Hold the receipts against the tills' own totals. Find the lines a join dropped |
| 6. A normal day | Find the store that sent nothing. Work out what a normal day is |
| 7. The false alarm | A holiday is not an error. See what silence costs |
| 8. The production version | The same checks as a tested program, and what is still wrong |

**You need:** the warehouse running. From the project folder: `docker compose up -d warehouse`.

**About ten minutes.**

In [1]:
from lab_helpers import bonus, keep, lab_copy, money, plant, prepare_week, production, show_check, sql, true_net_sales

prepare_week()

The bonus week is loaded: seven nights, Mon 24 to Sun 30 August. 24,384 sales lines.
The five standard checks ran on every night: 35 of 35 passed.


## Step 1: The best Thursday ever

**Samir Haddad** manages `S05`, the smallest store. Here are his last five Thursdays.

In [2]:
sql("""
    SELECT sales_date, receipts, lines, net_sales_cents
    FROM output.daily_store_sales
    WHERE store_id = 'S05' AND extract(isodow FROM sales_date) = 4
    ORDER BY sales_date
""")

,sales_date,receipts,lines,net_sales_cents
0,2026-07-30,25,78,44032
1,2026-08-06,25,88,56427
2,2026-08-13,28,82,64767
3,2026-08-20,27,84,57865
4,2026-08-27,29,84,210152


Four Thursdays between $440 and $650. Then 210152 cents: $2,101.52, with a normal number of receipts. Look at the largest lines of that day.

In [3]:
sql("""
    SELECT line_id, product_name, unit, quantity, unit_price_cents, amount_cents
    FROM clean.sales_lines
    WHERE store_id = 'S05' AND sales_date = DATE '2026-08-27'
    ORDER BY amount_cents DESC
    LIMIT 4
""")

,line_id,product_name,unit,quantity,unit_price_cents,amount_cents
0,S05-0827-000020,Bananas,kg,1180,129,152220
1,S05-0827-000047,Olive oil 500ml,each,3,949,2847
2,S05-0827-000012,Salmon fillet,each,2,1195,2390
3,S05-0827-000028,Salmon fillet,each,2,1195,2390


The first line: **1,180 kilograms of bananas**, for 152220 cents. $1,522.20, on one receipt.

Nobody carried 1,180 kg of bananas to a till. The scale weighed 1.180 kg, and it sends its weight in grams. Somewhere a 1,180 was written where 1.180 belonged.

In [4]:
reported = 210152
right = true_net_sales("S05", "2026-08-27")
print("The pipeline says:    ", money(reported))
print("The answer key says:  ", money(right))
print("Too high by:          ", money(reported - right))
print("Samir is overpaid by: ", money(bonus(reported) - bonus(right)))

The pipeline says:     $2,101.52
The answer key says:   $580.84
Too high by:           $1,520.68
Samir is overpaid by:  $15.21


One line of bananas is worth more than the store's whole day. Samir's bonus is $15.21 too high.

Every standard check passed. The quantity is a number. The amount is a number. The total is positive.

## Step 2: The common version

The usual fix is a quick rule on the number: "a quantity is above zero". Count the rows that would fail.

In [5]:
sql("""
    SELECT line_type, count(*) AS lines_that_would_fail
    FROM clean.sales_lines
    WHERE quantity <= 0
    GROUP BY line_type
""")

,line_type,lines_that_would_fail
0,refund,89


89 rows fail, and all 89 are refunds. A refund has a negative quantity on purpose. That rule fails on good rows, and it does not even look at the bananas: 1,180 is above zero.

## Step 3: A second common version

Try a limit on the money instead: "no line is above $200".

In [6]:
sql("""
    SELECT line_id, store_id, sales_date, product_name, quantity, amount_cents
    FROM clean.sales_lines
    WHERE amount_cents > 20000
""")

,line_id,store_id,sales_date,product_name,quantity,amount_cents
0,S05-0827-000020,S05,2026-08-27,Bananas,1180,152220
1,S02-0829-000018,S02,2026-08-29,Still water 500ml,400,23600


Two rows. The bananas, and 400 bottles of still water for $236.00 at store `S02`.

The water is **real**. A café bought 400 bottles on Saturday morning. If your check fires on it, somebody phones the store, the store says "yes, we sold that", and the next alarm from this check is read a little later.

One money limit for every product cannot tell a wrong weight from a big order.

## Step 4: The careful version

The fault is about **weight**. So look at the products that are sold by weight, apart from the ones sold by the piece.

In [7]:
sql("""
    SELECT unit, count(*) AS lines, min(abs(quantity)) AS smallest, max(abs(quantity)) AS largest
    FROM clean.sales_lines
    GROUP BY unit
    ORDER BY unit
""")

,unit,lines,smallest,largest
0,each,20749,1.000,400
1,kg,3635,0.201,1180


In [8]:
sql("""
    SELECT line_id, store_id, sales_date, product_name, quantity
    FROM clean.sales_lines
    WHERE unit = 'kg'
    ORDER BY quantity DESC
    LIMIT 4
""")

,line_id,store_id,sales_date,product_name,quantity
0,S05-0827-000020,S05,2026-08-27,Bananas,1180.000
1,S01-0817-000126,S01,2026-08-17,Bananas,1.999
2,S01-0823-000015,S01,2026-08-23,Apples,1.999
3,S04-0807-000072,S04,2026-08-07,Onions,1.997


Among 3,635 lines sold by weight, the largest real one is just under 2 kg. Then comes 1,180.

So a believable weight has a top. Lotus's rule: **not more than 20 kg of one product on one line**. That is ten times the largest real weight, so no real shopper hits it.

Goods sold by the piece get a different rule: the quantity is a whole number. Nobody buys half a bottle. And 400 bottles is a whole number.

In [9]:
BELIEVABLE = """
    SELECT line_id, store_id, sales_date, product_name, unit, quantity, amount_cents
    FROM clean.sales_lines
    WHERE (unit = 'kg' AND abs(quantity) > 20)
       OR (unit = 'each' AND quantity <> round(quantity))
"""
yours = sql(BELIEVABLE)
yours

,line_id,store_id,sales_date,product_name,unit,quantity,amount_cents
0,S05-0827-000020,S05,2026-08-27,Bananas,kg,1180,152220


In [10]:
keep("believable_quantity", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (A weight is a believable weight)
The same rows.


One row: the bananas. The 400 bottles pass.

A check that a number is between a smallest and a largest believable value is called a **range check**. The work is choosing the range, and it is rarely one range for the whole column. Here it depends on the unit.

Be honest about the limit. A cashier who types 15 kg where 1.5 kg was meant passes this check. A limit of 20 catches the grams mistake. It does not catch every wrong weight.

## Step 5: The shape of a text

A number has a range. A text has a **shape**. Look at a few receipt ids.

In [11]:
sql("""
    SELECT DISTINCT store_id, till_id, receipt_id
    FROM clean.sales_lines
    WHERE sales_date = DATE '2026-08-24' AND store_id IN ('S03', 'S04')
    ORDER BY receipt_id
    LIMIT 4
""")

,store_id,till_id,receipt_id
0,S03,T1,S03-T1-004491
1,S03,T1,S03-T1-004492
2,S03,T1,S03-T1-004493
3,S03,T1,S03-T1-004494


The shape is: the line's own store, its own till, and six digits. A refund must find the receipt it refunds by this id, so a damaged id breaks refunds.

Write it as a check. `translate(..., '0123456789', '')` removes every digit, so what is left must be nothing.

In [12]:
RIGHT_SHAPE = """
    SELECT line_id, store_id, sales_date, receipt_id
    FROM clean.sales_lines
    WHERE length(receipt_id) <> 13
       OR substr(receipt_id, 1, 7) <> store_id || '-' || till_id || '-'
       OR translate(substr(receipt_id, 8, 6), '0123456789', '') <> ''
"""
yours = sql(RIGHT_SHAPE)
keep("right_shape", yours)

Your query:           0 bad rows
The production check: 0 bad rows  (A receipt id has the right shape)
The same rows.


Every id has the right shape. See the check fail once, on the private copy: one id loses its dashes.

In [13]:
lab_copy()
plant("UPDATE lab.sales_lines SET receipt_id = 'S03T1004493' WHERE line_id = 'S03-0824-000008'")
production("right_shape", on_the_lab=True)

lab.sales_lines is a fresh copy of clean.sales_lines: 24,384 rows.


Planted in lab.sales_lines.


,line_id,store_id,sales_date,receipt_id
0,S03-0824-000008,S03,2026-08-24,S03T1004493


A check that a text value has the right shape is called a **format check**.

## Step 6: A line must add up

One more check uses nothing but the line itself. Every line has a quantity, a price and an amount. The amount should be the quantity times the price.

In [14]:
ADDS_UP = """
    SELECT line_id, store_id, sales_date, quantity, unit_price_cents, amount_cents,
           CAST(round(quantity * unit_price_cents) AS BIGINT) AS should_be_cents
    FROM clean.sales_lines
    WHERE amount_cents <> CAST(round(quantity * unit_price_cents) AS BIGINT)
"""
yours = sql(ADDS_UP)
yours

,line_id,store_id,sales_date,quantity,unit_price_cents,amount_cents,should_be_cents
0,S03-0824-000008,S03,2026-08-24,-1,949,949,-949


In [15]:
keep("line_adds_up", yours)

Your query:           1 bad rows
The production check: 1 bad rows  (A line adds up)
The same rows.


One row, and you know it: the refund from milestone 1. Minus one bottle of olive oil at 949 should be −949, and the line says +949. This check found it without knowing what a refund is.

But it did **not** find the bananas. On that line, 1,180 × 129 really is 152,220. The line adds up. It is the weight that is wrong.

| Check | Finds the bananas | Finds the refund | Fires on the 400 bottles |
|---|---|---|---|
| Quantity above zero | No | No | No, but on 89 good refunds |
| No line above $200 | Yes | No | **Yes** |
| A believable weight, per unit | **Yes** | No | No |
| A line adds up | No | **Yes** | No |

No single check sees everything. Each one sees one kind of wrong.

## What you now know

- A range check tests that a number is between a smallest and a largest believable value.
- One range for a whole column is usually wrong. Choose it per group: here, per unit.
- A format check tests the shape of a text, such as a receipt id.
- A limit catches some faults and lets others through. Say which.

**Say it in an interview:** "A scale sent a weight in grams, so one line said 1,180 kg of bananas: $1,522, more than the store's whole day. A flat 'no line above $200' rule would have caught it, and also a real order of 400 bottles of water. I used a range per unit instead: at most 20 kg on a line for goods sold by weight, whole numbers for goods sold by the piece."

**The theory:** [Data Quality & Validation](https://www.d8loop.com/modules/knowledge-corner/data-pipeline/data-quality-and-validation), section "What is good data?", the question "Valid?".

Next: [notebook 4](04_rules_of_the_business.ipynb). Some rules are not in the data at all. A person has to tell you.